# 2D Image & Grid Convolution Recipe

This recipe demonstrates how to perform **2D Image Filtering** and **Mathematical Morphology** using `algebrax.transforms.convolve`.

## Mathematical Foundation

1. **2D Spatial Mappings**: An image is a 2D sparse vector $f[(r, c)] = \text{intensity}$, and a kernel is $g[(dr, dc)] = \text{weight}$.
2. **2D Key Addition**: Passing `key_op = lambda p1, p2: (p1[0] + p2[0], p1[1] + p2[1])` shifts coordinates: $(r, c) + (dr, dc) = (r + dr, c + dc)$.
3. **Semiring Generalization**:
   - **Standard Semiring** $(+, \times)$: Linear 2D Spatial Filtering (Sobel, Blur, Sharpen)
     $$h[r, c] = \sum_{dr, dc} f[r - dr, c - dc] \cdot g[dr, dc]$$
   - **Arctic / Max-Plus Semiring** $(\max, +)$: Morphological Dilation (Max-Pooling)
     $$h[r, c] = \max_{dr, dc} (f[r - dr, c - dc] + g[dr, dc])$$

In [ ]:
from collections.abc import Mapping

import algebrax as ax


def add_2d(p1: tuple[int, int], p2: tuple[int, int]) -> tuple[int, int]:
    return (p1[0] + p2[0], p1[1] + p2[1])


def print_image(img: Mapping[tuple[int, int], float], title: str, rows: int = 8, cols: int = 8):
    print(f'\n--- {title} ---')
    for r in range(rows):
        line = []
        for c in range(cols):
            v = img.get((r, c), 0.0)
            char = '██' if v > 0.7 else ('▒▒' if v > 0.3 else ('--' if v < -0.3 else '  '))
            line.append(char)
        print(''.join(line))

## 1. Linear Edge Detection (Standard Semiring)

We create a synthetic 8x8 cross pattern image and apply a 3x3 **Sobel Horizontal Filter**.

In [ ]:

synthetic_image = {(r, c): 1.0 for r in range(8) for c in range(8) if r in (3, 4) or c in (3, 4)}

sobel_h = {
    (-1, -1): -1.0,
    (-1, 0): 0.0,
    (-1, 1): 1.0,
    (0, -1): -2.0,
    (0, 0): 0.0,
    (0, 1): 2.0,
    (1, -1): -1.0,
    (1, 0): 0.0,
    (1, 1): 1.0,
}

sobel_result = ax.transforms.convolve(synthetic_image, sobel_h, key_op=add_2d, semiring=ax.semiring.StandardSemiring())

print_image(synthetic_image, 'Original Cross Pattern')
print_image(sobel_result, 'Sobel Horizontal Edge Response')

## 2. Morphological Dilation (Arctic / Max-Plus Semiring)

By using the **Arctic Semiring** $(\max, +)$, convolution performs **Morphological Dilation**, expanding shape boundaries.

In [ ]:
import algebrax as ax

dilation_kernel = {
    (-1, 0): 0.0,
    (0, -1): 0.0,
    (0, 0): 0.0,
    (0, 1): 0.0,
    (1, 0): 0.0,
}

dilated = ax.transforms.convolve(synthetic_image, dilation_kernel, key_op=add_2d, semiring=ax.semiring.ArcticSemiring())

print_image(dilated, 'Morphological Dilation Output (Max-Plus)')

## 3. Pillow Image Ingestion & Sharpening

We can ingest PIL Image objects directly into `algebrax` sparse 2D vectors and run sharpening filters.

In [ ]:
from PIL import Image

import algebrax as ax

im = Image.new('L', (16, 16), color=0)
for r in range(4, 12):
    for c in range(4, 12):
        im.putpixel((c, r), 255)

sparse_img = {(r, c): im.getpixel((c, r)) / 255.0 for r in range(16) for c in range(16) if im.getpixel((c, r)) > 0}

sharpen_kernel = {(-1, 0): -1.0, (0, -1): -1.0, (0, 0): 5.0, (0, 1): -1.0, (1, 0): -1.0}
sharpened = ax.transforms.convolve(sparse_img, sharpen_kernel, key_op=add_2d, semiring=ax.semiring.StandardSemiring())

print_image(sparse_img, 'PIL 16x16 Input Image', rows=16, cols=16)
print_image(sharpened, 'Sharpened Output Image', rows=16, cols=16)